In [ ]:
# ==========================================
# INSTALLATIONS (Run this in a separate cell if using Colab)
# !pip install pycountry tiktoken transformers statsmodels matplotlib seaborn scipy
# ==========================================

import os
import json
import unicodedata
import requests
import numpy as np
import pandas as pd
import pycountry
import tiktoken
from transformers import AutoTokenizer
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: If running in Colab, this pulls your Hugging Face token safely.
# If running locally, make sure you set the HF_TOKEN environment variable.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except ImportError:
    pass

# ==========================================
# PHASE 1: FREQUENCY PROFILES & PARSERS
# ==========================================
class FrequencyProfile:
    """
    Manages language-specific character frequency profiles with local caching.
    Ensures early NFC normalization and OOV penalties. Unified under Leipzig.
    """
    def __init__(self, locale: str, epsilon: float = 1e-7, cache_dir: str = "./cache"):
        self.locale = locale
        self.epsilon = epsilon
        self.cache_dir = cache_dir
        self.frequency_map = {}

        os.makedirs(self.cache_dir, exist_ok=True)
        self._load_and_normalize_profile()

    def _load_and_normalize_profile(self) -> None:
        self.frequency_map = self._read_leipzig_collection()

    def _read_leipzig_collection(self) -> dict:
        cache_path = os.path.join(self.cache_dir, f"{self.locale}_leipzig_freq.json")
        if os.path.exists(cache_path):
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)

        # Points directly to the local repository folder
        raw_filepath = f"./raw_corpora/{self.locale}_letters.txt"
        if not os.path.exists(raw_filepath):
            raise FileNotFoundError(f"Raw corpus missing: {raw_filepath}")

        aggregated_counts = {}
        with open(raw_filepath, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 3 and parts[0].isdigit():
                    raw_char = parts[1]
                    count = int(parts[2])
                    if len(raw_char) != 1:  # Skip digraphs
                        continue
                    normalized_char = unicodedata.normalize('NFC', raw_char)
                    aggregated_counts[normalized_char] = aggregated_counts.get(normalized_char, 0) + count

        total_obs = sum(aggregated_counts.values())
        frequency_map = {char: (count / total_obs) for char, count in aggregated_counts.items()}

        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(frequency_map, f, ensure_ascii=False, indent=2)
        return frequency_map

    def get_rarity_coefficient(self, character: str) -> float:
        normalized_char = unicodedata.normalize('NFC', character)
        return self.frequency_map.get(normalized_char, self.epsilon)

# ==========================================
# PHASE 2: CLDR INGESTION & DATA ENGINEERING
# ==========================================
def get_valid_iso_alpha2_codes():
    return {country.alpha_2 for country in pycountry.countries}

def fetch_cldr_territories(locale):
    url = f"https://raw.githubusercontent.com/unicode-org/cldr-json/main/cldr-json/cldr-localenames-full/main/{locale}/territories.json"
    response = requests.get(url)
    return response.json()['main'][locale]['localeDisplayNames']['territories'] if response.status_code == 200 else None

def normalize_string(text):
    return unicodedata.normalize('NFC', text) if text else None

def build_robust_localization_matrix(locales):
    print("Ingesting CLDR data...")
    iso_whitelist = get_valid_iso_alpha2_codes()
    english_raw = fetch_cldr_territories('en')

    master_data = {
        code: {'en': normalize_string(english_raw.get(code))}
        for code in iso_whitelist if english_raw.get(code)
    }

    missing_data_logs = {locale: 0 for locale in locales if locale != 'en'}

    for locale in locales:
        if locale == 'en': continue
        localized_raw = fetch_cldr_territories(locale)
        if not localized_raw:
            continue

        for code in master_data.keys():
            name = localized_raw.get(code)
            if name:
                master_data[code][locale] = normalize_string(name)
                master_data[code][f"{locale}_is_fallback"] = False
            else:
                missing_data_logs[locale] += 1
                master_data[code][locale] = master_data[code]['en']
                master_data[code][f"{locale}_is_fallback"] = True

    df = pd.DataFrame.from_dict(master_data, orient='index')
    df.index.name = 'iso_alpha2'
    return df

# ==========================================
# PHASE 3: MATHEMATICAL CLOUS CALCULATION
# ==========================================
def compute_clous_dataframe(country_matrix_df, profiles_dict, target_locales):
    print("Calculating CLOUS scores...")
    clous_results = {}

    for iso_code, row in country_matrix_df.iterrows():
        clous_results[iso_code] = {}
        for locale in target_locales:
            text = row[locale]
            profile = profiles_dict[locale]

            if not text or pd.isna(text):
                clous_results[iso_code][locale] = np.nan
                continue

            surprisal_sum = 0.0
            N = len(text)

            for char in text:
                freq = profile.get_rarity_coefficient(char)
                surprisal_sum += np.log2(max(freq, profile.epsilon))

            clous_results[iso_code][locale] = -(surprisal_sum / N)

    return pd.DataFrame.from_dict(clous_results, orient='index')

# ==========================================
# PHASE 4: TOKENIZATION & STATISTICAL ANALYSIS
# ==========================================
TOKENIZER_REGISTRY = {
    "gpt4o":    {"backend": "tiktoken", "model": "o200k_base"},
    "gpt4":     {"backend": "tiktoken", "model": "cl100k_base"},
    "llama3":   {"backend": "transformers", "model": "meta-llama/Meta-Llama-3.1-8B"},
    "gemma":    {"backend": "transformers", "model": "google/gemma-7b"},
    "xlmr":     {"backend": "transformers", "model": "xlm-roberta-base"},
    "mbert":    {"backend": "transformers", "model": "bert-base-multilingual-cased"},
}

def load_tokenizer(name: str):
    config = TOKENIZER_REGISTRY[name]
    if config["backend"] == "tiktoken":
        return tiktoken.get_encoding(config["model"])
    else:
        return AutoTokenizer.from_pretrained(config["model"])

def compute_fertility(tokenizer, text: str, backend: str) -> float:
    if backend == "tiktoken":
        token_count = len(tokenizer.encode(text))
    else:
        token_count = len(tokenizer.encode(text, add_special_tokens=False))
    return token_count / len(text)

def run_tokenization_benchmark(country_matrix_df, clous_scores_df, target_locales):
    print("Running tokenization benchmarks...")
    results = []

    for tokenizer_name, config in TOKENIZER_REGISTRY.items():
        print(f"  -> Testing {tokenizer_name}...")
        try:
            tok = load_tokenizer(tokenizer_name)
        except Exception as e:
            print(f"Skipping {tokenizer_name} due to load error (HF token missing?): {e}")
            continue

        for iso_code, row in country_matrix_df.iterrows():
            for locale in target_locales:
                text = row[locale]
                is_fallback = row.get(f"{locale}_is_fallback", False)
                fertility = compute_fertility(tok, text, config["backend"])
                clous = clous_scores_df.loc[iso_code, locale]

                results.append({
                    "iso_alpha2": iso_code,
                    "locale": locale,
                    "tokenizer": tokenizer_name,
                    "text": text,
                    "is_fallback": is_fallback,
                    "fertility_score": fertility,
                    "clous_score": clous,
                })

    return pd.DataFrame(results)

def analyze_clous_fertility_correlation(results_df):
    print("Running statistical correlations...")
    correlation_results = []

    for tokenizer in results_df['tokenizer'].unique():
        for locale in results_df['locale'].unique():
            subset = results_df[
                (results_df['tokenizer'] == tokenizer) &
                (results_df['locale'] == locale) &
                (results_df['is_fallback'] == False)
            ].dropna(subset=['clous_score', 'fertility_score'])

            if len(subset) < 30: continue

            # Suppress constant input warning for zh-Hans x mBERT
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                spearman_r, spearman_p = stats.spearmanr(subset['clous_score'], subset['fertility_score'])
                kendall_tau, kendall_p = stats.kendalltau(subset['clous_score'], subset['fertility_score'])

            correlation_results.append({
                'tokenizer': tokenizer,
                'locale': locale,
                'n': len(subset),
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'kendall_tau': kendall_tau,
                'kendall_p': kendall_p
            })

    corr_df = pd.DataFrame(correlation_results)

    # Benjamini-Hochberg Correction (NaN-safe)
    if not corr_df.empty:
        valid_idx = corr_df['spearman_p'].notna()
        if valid_idx.any():
            _, corrected_p, _, _ = multipletests(corr_df.loc[valid_idx, 'spearman_p'], method='fdr_bh')
            corr_df['spearman_p_corrected'] = float('nan')
            corr_df.loc[valid_idx, 'spearman_p_corrected'] = corrected_p
            corr_df['significant_corrected'] = corr_df['spearman_p_corrected'] < 0.05

    return corr_df

# ==========================================
# PHASE 5: CHART & TABLE GENERATION
# ==========================================
def generate_correlation_heatmap():
    print("Generating Figure 1 (Heatmap)...")
    tokenizers = ['Gemma-7B', 'GPT-4', 'GPT-4o', 'Llama 3.1', 'mBERT', 'XLM-R']
    languages = [
        'English (en)', 'Spanish (es)', 'French (fr)', 'Turkish (tr)',
        'Russian (ru)', 'Arabic (ar)', 'Hindi (hi)', 'Tamil (ta)',
        'Korean (ko)', 'Mandarin (zh-Hans)', 'Japanese (ja)'
    ]

    rho_values = [
        [0.413, 0.250, 0.316, 0.229, 0.494, 0.372],
        [0.314, 0.396, 0.365, 0.405, 0.418, 0.429],
        [0.265, 0.320, 0.383, 0.316, 0.306, 0.360],
        [0.177, 0.201, 0.186, 0.136, 0.170, 0.280],
        [0.387, 0.404, 0.432, 0.356, 0.338, 0.407],
        [0.212, 0.066, 0.227, 0.142, 0.038, 0.174],
        [0.165, 0.491, 0.120, 0.103, 0.142, 0.202],
        [0.315, 0.162, 0.282, 0.162, 0.279, 0.247],
        [0.161, 0.624, 0.422, 0.449, 0.174, 0.405],
        [0.311, 0.614, 0.494, 0.465, np.nan, 0.339],
        [0.400, 0.686, 0.437, 0.400, 0.389, 0.319],
    ]

    sig_values = [
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, False, True, True, False, True],
        [True, True, False, False, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, False, True],
        [True, True, True, True, True, True],
    ]

    df_rho = pd.DataFrame(rho_values, index=languages, columns=tokenizers)

    annot_data = []
    for i in range(len(languages)):
        row_annots = []
        for j in range(len(tokenizers)):
            val = rho_values[i][j]
            is_sig = sig_values[i][j]
            if pd.isna(val):
                row_annots.append("—")
            else:
                star = " ★" if is_sig else ""
                row_annots.append(f"{val:.3f}{star}")
        annot_data.append(row_annots)

    annot_df = pd.DataFrame(annot_data, index=languages, columns=tokenizers)

    sns.set_theme(style="white")
    plt.figure(figsize=(10, 8))

    ax = sns.heatmap(
        df_rho,
        annot=annot_df,
        fmt="",
        cmap="Blues",
        cbar_kws={'label': 'Spearman Correlation (ρ)'},
        vmin=0.0, vmax=0.7,
        linewidths=1,
        linecolor='white',
        square=False
    )

    ax.hlines([4], *ax.get_xlim(), colors='black', linewidth=2.5)
    ax.text(-0.8, 2, 'Latin\nScripts', va='center', ha='right', fontsize=12, fontweight='bold', color='#333333')
    ax.text(-0.8, 7.5, 'Non-Latin\nScripts', va='center', ha='right', fontsize=12, fontweight='bold', color='#333333')

    plt.title("Figure 1. Spearman ρ Correlation: CLOUS vs. Tokenization Fertility", fontsize=14, pad=20, fontweight='bold')
    plt.xlabel("Tokenizer Architecture", fontsize=12, labelpad=10)
    plt.ylabel("", fontsize=12)

    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)

    plt.tight_layout()
    output_filename = "./results/Figure_1_CLOUS_Heatmap.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close()

def generate_scatter_plots():
    print("Generating Figure 2 (Scatter Plots)...")
    data_path = './cache/benchmark_results.csv'
    df = pd.read_csv(data_path)

    df = df[df['is_fallback'] == False].dropna(subset=['clous_score', 'fertility_score'])

    script_map = {
        'en': 'Latin', 'es': 'Latin', 'fr': 'Latin',
        'tr': 'Latin+', 'ru': 'Cyrillic', 'ar': 'Arabic',
        'hi': 'Devanagari', 'ta': 'Tamil', 'ko': 'Hangul',
        'zh-Hans': 'Logographic', 'ja': 'Mixed-script'
    }
    df['Script Family'] = df['locale'].map(script_map)
    df = df[~((df['tokenizer'] == 'mbert') & (df['locale'] == 'zh-Hans'))]

    sns.set_theme(style="whitegrid", font_scale=1.1)

    g = sns.FacetGrid(
        df,
        col="tokenizer",
        col_wrap=3,
        height=4,
        aspect=1.2,
        sharex=False,
        sharey=True
    )

    g.map_dataframe(
        sns.scatterplot,
        x="clous_score",
        y="fertility_score",
        hue="Script Family",
        palette="tab10",
        alpha=0.7,
        edgecolor=None,
        s=40
    )

    g.map_dataframe(
        sns.regplot,
        x="clous_score",
        y="fertility_score",
        scatter=False,
        color='black',
        line_kws={"linestyle": "--", "linewidth": 2, "alpha": 0.8}
    )

    def annotate_corr(data, **kws):
        if len(data) < 2: return
        r, p = stats.spearmanr(data['clous_score'], data['fertility_score'])
        ax = plt.gca()
        p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        text = f"ρ = {r:.3f}\n{p_text}"
        ax.text(0.05, 0.95, text, transform=ax.transAxes, ha="left", va="top",
                fontsize=11, fontweight='bold',
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9))

    g.map_dataframe(annotate_corr)
    g.set_axis_labels("CLOUS Score (Bits/Char)", "Fertility Score (Tokens/Char)")
    g.set_titles(col_template="{col_name}", size=14, weight='bold')
    g.add_legend(title="Script Family", title_fontsize='13', fontsize='11', bbox_to_anchor=(1.02, 0.5), loc='center left')

    g.figure.subplots_adjust(top=0.9)
    g.figure.suptitle(
        "Architectural Scaling vs. Orthographic Variance\nTokenization Fragmentation by Script Family",
        fontsize=16, fontweight='bold', y=0.98
    )

    output_filename = "./results/Figure_2_Stratified_Scatter.png"
    g.figure.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.close()

def generate_outlier_table():
    print("Generating Table 3 (Outliers)...")
    clous_path = './cache/clous_scores.csv'
    bench_path = './cache/benchmark_results.csv'

    clous_df = pd.read_csv(clous_path, index_col=0)
    bench_df = pd.read_csv(bench_path)

    mean_clous = clous_df.mean(axis=1)
    valid_bench = bench_df[bench_df['is_fallback'] == False]

    fert_pivot = valid_bench.pivot_table(
        index='iso_alpha2',
        columns='tokenizer',
        values='fertility_score',
        aggfunc='mean'
    )

    outlier_df = pd.DataFrame({'Mean CLOUS': mean_clous})
    outlier_df = outlier_df.join(fert_pivot[['mbert', 'gpt4o']])
    outlier_df.rename(columns={
        'mbert': 'Mean Fertility (mBERT)',
        'gpt4o': 'Mean Fertility (GPT-4o)'
    }, inplace=True)

    outlier_df['Δ Fertility'] = outlier_df['Mean Fertility (mBERT)'] - outlier_df['Mean Fertility (GPT-4o)']
    top_15 = outlier_df.sort_values(by='Mean CLOUS', ascending=False).head(15).reset_index()
    top_15.rename(columns={'index': 'ISO Code'}, inplace=True)

    def get_country_name(iso_code):
        try:
            return pycountry.countries.get(alpha_2=iso_code.upper()).name
        except:
            return iso_code.upper()

    top_15['Country Name'] = top_15['ISO Code'].apply(get_country_name)
    top_15['Script Families'] = "Mixed"

    final_cols = ['ISO Code', 'Country Name', 'Mean CLOUS', 'Script Families',
                  'Mean Fertility (mBERT)', 'Mean Fertility (GPT-4o)', 'Δ Fertility']
    top_15 = top_15[final_cols]

    top_15['Mean CLOUS'] = top_15['Mean CLOUS'].round(3)
    top_15['Mean Fertility (mBERT)'] = top_15['Mean Fertility (mBERT)'].round(2)
    top_15['Mean Fertility (GPT-4o)'] = top_15['Mean Fertility (GPT-4o)'].round(2)
    top_15['Δ Fertility'] = top_15['Δ Fertility'].round(2)

    output_filename = "./results/Table_3_Djibouti_Effect.csv"
    top_15.to_csv(output_filename, index=False)

# ==========================================
# EXECUTION LAYER
# ==========================================
if __name__ == "__main__":
    # Ensure all required directories exist
    os.makedirs("./raw_corpora", exist_ok=True)
    os.makedirs("./cache", exist_ok=True)
    os.makedirs("./results", exist_ok=True)

    target_bcp47_tags = ['en', 'es', 'fr', 'ru', 'ar', 'hi', 'zh-Hans', 'ja', 'ta', 'tr', 'ko']

    # 1. Ingest Data
    country_matrix_df = build_robust_localization_matrix(target_bcp47_tags)

    # 2. Load Profiles
    print("Loading Frequency Profiles...")
    profiles_dict = {loc: FrequencyProfile(loc) for loc in target_bcp47_tags}

    # 3. Calculate CLOUS
    clous_scores_df = compute_clous_dataframe(country_matrix_df, profiles_dict, target_bcp47_tags)

    # 4. Tokenization Benchmarks
    benchmark_results = run_tokenization_benchmark(country_matrix_df, clous_scores_df, target_bcp47_tags)

    # 5. Statistical Output
    final_stats = analyze_clous_fertility_correlation(benchmark_results)

    # 6. Save Data Caches
    benchmark_results.to_csv('./cache/benchmark_results.csv', index=False)
    final_stats.to_csv('./cache/final_stats.csv', index=False)
    clous_scores_df.to_csv('./cache/clous_scores.csv')
    print("Data caching complete.")

    # 7. Generate Output Files
    generate_correlation_heatmap()
    generate_scatter_plots()
    generate_outlier_table()
    print("\nPipeline complete! All figures and tables are saved in the ./results/ directory.")